# 📝 Natural Language Processing with Python

*A hands-on tutorial covering the full NLP pipeline — from raw text to classification and embeddings.*

---

## 🎯 What You'll Learn

| Part | Topic | Key Skills |
|------|-------|------------|
| **1** | Text Preprocessing | Tokenization, stopword removal, stemming, cleaning pipelines |
| **2** | Text Representation | Bag-of-Words, TF-IDF, feature matrices |
| **3** | Text Classification | Naive Bayes, Logistic Regression, sklearn Pipelines |
| **4** | Word Embeddings | PyTorch `nn.Embedding`, cosine similarity, PCA visualization |
| **5** | Transformers (Conceptual) | Self-attention, BERT vs GPT, Hugging Face ecosystem |
| **6** | Wrap-up & Exercises | Practice problems, recommended resources |

> **Prerequisites:** Basic Python, familiarity with NumPy/pandas. No prior NLP experience needed.
>
> **Environment:** All data is **synthetic** — no downloads required.

## 📖 Introduction to NLP

**Natural Language Processing (NLP)** is the branch of artificial intelligence that gives computers the ability to understand, interpret, and generate human language.

### Why NLP Matters

Every day, the world generates enormous amounts of unstructured text:
- Emails, chat messages, social-media posts
- Product reviews, news articles, medical records
- Legal documents, patents, scientific papers

NLP techniques let us **extract meaning** from this data at scale.

### Real-World Applications

| Application | Example |
|-------------|--------|
| **Sentiment Analysis** | Classify product reviews as positive or negative |
| **Machine Translation** | Google Translate, DeepL |
| **Chatbots & Virtual Assistants** | Siri, Alexa, ChatGPT |
| **Named Entity Recognition** | Extract people, places, dates from news |
| **Text Summarization** | Condense long articles into key points |
| **Spam Detection** | Filter unwanted emails |

In this notebook we'll build the foundational skills — cleaning text, converting it to numbers, and training classifiers — before touching on modern deep-learning methods.

---

# Part 1 — Text Preprocessing

> Raw text is messy. Before any model can work with language, we need to **normalize**, **tokenize**, and **clean** it.

---

In [ ]:
# Cell 1: Imports
import re
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
)

import torch
import torch.nn as nn

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print('All imports successful.')

### 📄 Sample Text Data

We'll use **synthetic movie reviews** so the notebook is fully self-contained — no file downloads required.
Each review is labelled `positive` or `negative`.

In [ ]:
# Cell 2: Synthetic Movie Reviews
reviews = [
    # Positive reviews
    "An absolutely wonderful film with stunning visuals and a gripping storyline.",
    "I loved every minute of this movie! The acting was superb and the ending was perfect.",
    "A masterpiece of modern cinema. The director really outdid himself this time.",
    "Heartwarming and beautifully shot. I laughed, I cried, I cheered.",
    "Incredible performances from the entire cast. A must-see for any film lover.",
    "The screenplay was brilliant and the soundtrack was hauntingly beautiful.",
    "One of the best films I have seen in years. Truly exceptional storytelling.",
    "A feel-good movie that leaves you smiling long after the credits roll.",
    "Visually breathtaking with a deeply moving narrative. Highly recommended!",
    "Outstanding direction and phenomenal acting. This film deserves every award.",
    "Charming, witty, and endlessly entertaining. A real crowd-pleaser.",
    "The plot twists were unexpected and thrilling. Kept me on the edge of my seat!",
    # Negative reviews
    "A terrible waste of time. The plot made no sense and the acting was wooden.",
    "I was bored out of my mind. Nothing interesting happens for two hours.",
    "Awful dialogue and cringe-worthy scenes. I walked out halfway through.",
    "The worst movie I have ever seen. Save your money and skip this one.",
    "Predictable, dull, and painfully slow. I could not wait for it to end.",
    "Poor special effects and a laughable script. A complete disaster.",
    "Disappointing sequel that fails to capture the magic of the original.",
    "The characters were one-dimensional and the story was uninspired.",
    "An incoherent mess with terrible pacing. Absolutely dreadful.",
    "Over-hyped and under-delivered. I expected so much more from this director.",
    "Bland performances and a forgettable plot. Not worth your time.",
    "Painfully long with no payoff. One of the biggest letdowns of the year.",
]

labels = ['positive'] * 12 + ['negative'] * 12

df = pd.DataFrame({"review": reviews, "sentiment": labels})
print(f"Dataset: {len(df)} reviews  ({df['sentiment'].value_counts().to_dict()})")
df.head(6)

Let's do a quick exploratory look at the review lengths.

In [ ]:
# Cell 3: Quick EDA — review length distribution
df['word_count'] = df['review'].apply(lambda x: len(x.split()))

fig, ax = plt.subplots(figsize=(8, 3.5))
for sentiment, color in [('positive', '#4CAF50'), ('negative', '#F44336')]:
    subset = df[df['sentiment'] == sentiment]
    ax.hist(subset['word_count'], bins=8, alpha=0.6, label=sentiment, color=color, edgecolor='white')
ax.set_xlabel('Word Count')
ax.set_ylabel('Number of Reviews')
ax.set_title('Review Length Distribution by Sentiment')
ax.legend()
plt.tight_layout()
plt.show()

print(df.groupby('sentiment')['word_count'].describe().round(1))

### ✂️ Tokenization

**Tokenization** splits text into individual units (tokens) — usually words.

We use a **regex-based tokenizer** for reliability.  The pattern `\b\w+\b` matches word-boundary-delimited sequences of alphanumeric characters.

In [ ]:
# Cell 4: Regex-Based Tokenization
def tokenize(text: str) -> list[str]:
    '''Split text into lowercase word tokens using regex.'''
    return re.findall(r'\b\w+\b', text.lower())

# Demo on a single review
sample = reviews[0]
tokens = tokenize(sample)
print(f'Original : {sample}')
print(f'Tokens   : {tokens}')
print(f'# Tokens : {len(tokens)}')

### 🛑 Stopword Removal

**Stopwords** are extremely common words (like *the*, *is*, *and*) that carry little semantic value.  Removing them reduces noise and dimensionality.

We define a **hardcoded** stopword list — no NLTK download needed.

In [ ]:
# Cell 5: Stopword Removal
STOPWORDS = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves",
    "you", "your", "yours", "yourself", "yourselves",
    "he", "him", "his", "himself", "she", "her", "hers", "herself",
    "it", "its", "itself", "they", "them", "their", "theirs", "themselves",
    "what", "which", "who", "whom", "this", "that", "these", "those",
    "am", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "having", "do", "does", "did", "doing",
    "a", "an", "the", "and", "but", "if", "or", "because", "as",
    "until", "while", "of", "at", "by", "for", "with", "about",
    "against", "between", "through", "during", "before", "after",
    "above", "below", "to", "from", "up", "down", "in", "out",
    "on", "off", "over", "under", "again", "further", "then", "once",
    "here", "there", "when", "where", "why", "how", "all", "both",
    "each", "few", "more", "most", "other", "some", "such", "no",
    "nor", "not", "only", "own", "same", "so", "than", "too",
    "very", "s", "t", "can", "will", "just", "don", "should",
    "now", "d", "ll", "m", "o", "re", "ve", "y", "ain",
    "aren", "couldn", "didn", "doesn", "hadn", "hasn", "haven",
    "isn", "ma", "mightn", "mustn", "needn", "shan", "shouldn",
    "wasn", "weren", "won", "wouldn",
}

def remove_stopwords(tokens: list[str]) -> list[str]:
    return [t for t in tokens if t not in STOPWORDS]

# Demo
clean_tokens = remove_stopwords(tokens)
print(f'Before ({len(tokens):>2} tokens): {tokens}')
print(f'After  ({len(clean_tokens):>2} tokens): {clean_tokens}')

### 🌿 Stemming

**Stemming** reduces words to their root form by stripping common suffixes.

Below is a **simple rule-based stemmer** written in pure Python — no NLTK required.
It handles the most frequent English suffixes (`-ing`, `-ed`, `-ly`, `-tion`, `-s`).

In [ ]:
# Cell 6: Simple Suffix-Stripping Stemmer
def simple_stem(word: str) -> str:
    '''Reduce a word to an approximate stem by stripping common suffixes.'''
    if len(word) <= 3:
        return word
    suffixes = [
        ("ational", "ate"),
        ("tional", "tion"),
        ("fulness", "ful"),
        ("ousness", "ous"),
        ("iveness", "ive"),
        ("ingly", ""),
        ("ation", "ate"),
        ("ness", ""),
        ("ment", ""),
        ("able", ""),
        ("ible", ""),
        ("ally", "al"),
        ("ful", ""),
        ("ing", ""),
        ("ied", "y"),
        ("ies", "y"),
        ("ed", ""),
        ("ly", ""),
        ("er", ""),
        ("es", ""),
        ("s", ""),
    ]
    for suffix, replacement in suffixes:
        if word.endswith(suffix) and len(word) - len(suffix) >= 2:
            return word[: -len(suffix)] + replacement
    return word

# Demo
demo_words = ['running', 'played', 'beautifully', 'gripping', 'visuals',
              'performances', 'stunning', 'stories', 'acted', 'slowly']
for w in demo_words:
    print(f'  {w:>15s}  ->  {simple_stem(w)}')

### 📝 Lemmatization (Concept)

| | Stemming | Lemmatization |
|---|---|---|
| **Approach** | Rule-based suffix stripping | Dictionary / morphological analysis |
| **Output** | May not be a real word (*"studi"*) | Always a valid word (*"study"*) |
| **Speed** | Very fast | Slower (needs vocabulary lookup) |
| **Libraries** | Our custom stemmer, NLTK `PorterStemmer` | spaCy, NLTK `WordNetLemmatizer` |

> **For production NLP**, lemmatization (via [spaCy](https://spacy.io/) or NLTK with WordNet) generally produces better features, but stemming is a great lightweight starting point.

### 🔧 Text Cleaning Pipeline

Let's combine everything into a single `clean_text()` function.

In [ ]:
# Cell 7: Full Cleaning Pipeline
def clean_text(text: str, stem: bool = True) -> str:
    '''Tokenize, remove stopwords, optionally stem, and rejoin.'''
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    if stem:
        tokens = [simple_stem(t) for t in tokens]
    return ' '.join(tokens)

# Apply to the whole dataset
df['clean'] = df['review'].apply(clean_text)

print('Original vs. Cleaned')
print('=' * 70)
for i in [0, 4, 12, 18]:
    print(f"\nReview #{i} ({df.loc[i, 'sentiment']})")
    print(f"  Original : {df.loc[i, 'review']}")
    print(f"  Cleaned  : {df.loc[i, 'clean']}")

---

# Part 2 — Text Representation

> Machine-learning models need **numbers**, not words.  In this section we convert our cleaned text into numerical feature vectors.

---

### 📊 Bag of Words (BoW)

The **Bag-of-Words** model represents each document as a vector of word counts — ignoring order but capturing frequency.

We'll build one **manually** with `Counter`, then use scikit-learn's `CountVectorizer`.

In [ ]:
# Cell 8: Manual Bag of Words
sample_clean = df.loc[0, 'clean']
bow_manual = Counter(sample_clean.split())
print(f'Review #0 cleaned: "{sample_clean}"\n')
print('Manual BoW:')
for word, count in bow_manual.most_common(10):
    print(f'  {word:>15s} : {count}')

In [ ]:
# Cell 9: CountVectorizer BoW
count_vec = CountVectorizer()
X_bow = count_vec.fit_transform(df['clean'])

print(f'BoW matrix shape : {X_bow.shape}  (documents x unique terms)')
print(f'Vocabulary size   : {len(count_vec.vocabulary_)}')
print(f'\nSample terms: {list(count_vec.vocabulary_.keys())[:15]}')

# Show the BoW vector for the first review
bow_df = pd.DataFrame(X_bow.toarray(), columns=count_vec.get_feature_names_out())
print('\nBoW vector for review #0 (non-zero terms):')
row0 = bow_df.iloc[0]
print(row0[row0 > 0].to_dict())

### 📐 TF-IDF (Term Frequency – Inverse Document Frequency)

**TF-IDF** improves on raw counts by *down-weighting* words that appear in many documents (common, less informative) and *up-weighting* words that are distinctive to a few documents.

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\!\left(\frac{N}{\text{DF}(t)}\right)$$

| Symbol | Meaning |
|--------|--------|
| $\text{TF}(t,d)$ | How often term $t$ appears in document $d$ |
| $\text{DF}(t)$ | Number of documents containing $t$ |
| $N$ | Total number of documents |

In [ ]:
# Cell 10: TF-IDF Vectorization
tfidf_vec = TfidfVectorizer(max_features=80)
X_tfidf = tfidf_vec.fit_transform(df['clean'])

print(f'TF-IDF matrix shape: {X_tfidf.shape}')

# Top TF-IDF terms for the first positive & first negative review
feature_names = tfidf_vec.get_feature_names_out()
for idx, label in [(0, 'Positive'), (12, 'Negative')]:
    row = X_tfidf[idx].toarray().flatten()
    top_idx = row.argsort()[-5:][::-1]
    top_terms = [(feature_names[i], round(row[i], 3)) for i in top_idx]
    print(f'\n  Top TF-IDF terms for review #{idx} ({label}):')
    for term, score in top_terms:
        print(f'    {term:>15s}  {score:.3f}')

### 📈 Visualizations

Let's visualize **word frequencies** and **TF-IDF scores** to see what our features look like.

In [ ]:
# Cell 11: Word Frequency Bar Chart
all_tokens = ' '.join(df['clean']).split()
freq = Counter(all_tokens).most_common(15)
words, counts = zip(*freq)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(words[::-1], counts[::-1], color=sns.color_palette('muted', len(words)))
ax.set_xlabel('Frequency')
ax.set_title('Top 15 Words by Frequency (After Cleaning)')
for i, (w, c) in enumerate(zip(words[::-1], counts[::-1])):
    ax.text(c + 0.2, i, str(c), va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: TF-IDF Heatmap
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=feature_names)
top15 = tfidf_df.mean().nlargest(15).index.tolist()
heatmap_data = tfidf_df[top15].head(10).T  # first 10 docs

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=True, fmt='.2f',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'TF-IDF'})
ax.set_xlabel('Document Index')
ax.set_ylabel('Term')
ax.set_title('TF-IDF Scores (Top 15 Terms x First 10 Docs)')
plt.tight_layout()
plt.show()

---

# Part 3 — Text Classification (Sentiment Analysis)

> Now for the fun part: we'll train classifiers to **predict sentiment** from text features.

---

### 🏗️ Dataset Setup

We convert labels to binary, create TF-IDF features, and split into train/test sets.

In [ ]:
# Cell 13: Train / Test Split
y = (df['sentiment'] == 'positive').astype(int).values  # 1=positive, 0=negative

tfidf_full = TfidfVectorizer()
X_full = tfidf_full.fit_transform(df['clean'])

X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.25, random_state=SEED, stratify=y
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Train label distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}')
print(f'Test  label distribution: {dict(zip(*np.unique(y_test, return_counts=True)))}')

### 🧮 Naive Bayes Classifier

**Multinomial Naive Bayes** is a classic baseline for text classification.  It's fast, interpretable, and works surprisingly well on small datasets with sparse features.

In [ ]:
# Cell 14: Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

print('=== Naive Bayes - Classification Report ===\n')
print(classification_report(y_test, y_pred_nb, target_names=['negative', 'positive']))
print(f'Accuracy: {accuracy_score(y_test, y_pred_nb):.2%}')

### 📈 Logistic Regression

**Logistic Regression** is another strong baseline.  Unlike Naive Bayes it models feature interactions and often achieves higher accuracy on text data.

In [ ]:
# Cell 15: Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=SEED)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print('=== Logistic Regression - Classification Report ===\n')
print(classification_report(y_test, y_pred_lr, target_names=['negative', 'positive']))
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr):.2%}')

Let's examine which words the Logistic Regression model considers most indicative of positive and negative sentiment.

In [ ]:
# Cell 16: Most Informative Features (LR coefficients)
feature_names_full = tfidf_full.get_feature_names_out()
coefs = lr_model.coef_[0]

top_pos_idx = coefs.argsort()[-10:][::-1]
top_neg_idx = coefs.argsort()[:10]

print('Top 10 POSITIVE indicators:')
for idx in top_pos_idx:
    print(f'  {feature_names_full[idx]:>15s}  coef={coefs[idx]:+.3f}')

print('\nTop 10 NEGATIVE indicators:')
for idx in top_neg_idx:
    print(f'  {feature_names_full[idx]:>15s}  coef={coefs[idx]:+.3f}')

### 🔗 sklearn Pipeline + Cross-Validation

A `Pipeline` chains the vectorizer and classifier, making the workflow cleaner and preventing data leakage during cross-validation.

In [ ]:
# Cell 17: Pipeline with Cross-Validation
pipe = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', LogisticRegression(max_iter=1000, random_state=SEED)),
])

# Cross-validate on the ORIGINAL uncleaned reviews
cv_scores = cross_val_score(pipe, df['review'], y, cv=4, scoring='accuracy')

print('=== Pipeline Cross-Validation ===')
print(f"  Fold accuracies : {[f'{s:.2%}' for s in cv_scores]}")
print(f'  Mean accuracy   : {cv_scores.mean():.2%} +/- {cv_scores.std():.2%}')

### 📊 Confusion Matrix & Metrics Visualization

Let's visualize how our models perform on the test set.

In [ ]:
# Cell 18: Confusion Matrix Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, y_pred, name in [
    (axes[0], y_pred_nb, 'Naive Bayes'),
    (axes[1], y_pred_lr, 'Logistic Regression'),
]:
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['negative', 'positive'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}\nAccuracy: {accuracy_score(y_test, y_pred):.0%}')

plt.suptitle('Confusion Matrices - Test Set', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

# Part 4 — Word Embeddings (Conceptual + Demo)

---

### 🧠 What Are Word Embeddings?

Traditional methods (BoW, TF-IDF) treat words as **independent symbols** — they can't capture that *"good"* and *"great"* are semantically similar.

**Word embeddings** map each word to a **dense, low-dimensional vector** in a continuous space, where semantically related words are close together.

| Method | Year | Key Idea |
|--------|------|----------|
| **Word2Vec** | 2013 | Predict a word from its context (CBOW) or context from a word (Skip-gram) |
| **GloVe** | 2014 | Learn vectors from global word co-occurrence statistics |
| **FastText** | 2017 | Extend Word2Vec with sub-word (character n-gram) information |

**Why embeddings beat BoW:**
- Capture **semantic similarity** (king − man + woman ≈ queen)
- Much **lower dimensionality** (e.g., 300 vs. 50,000+)
- Serve as **pre-trained features** for downstream tasks

### 🔢 Simple Embedding Demo with PyTorch

Below we create a tiny vocabulary, assign each word a learnable embedding vector, and compute **cosine similarity** between word pairs.

In [ ]:
# Cell 19: PyTorch nn.Embedding Demo
# Build a small vocabulary from our reviews
all_words = sorted(set(' '.join(df['clean']).split()))
word2idx = {w: i for i, w in enumerate(all_words)}
vocab_size = len(word2idx)
embed_dim = 16  # small for demonstration

print(f'Vocabulary size: {vocab_size}')
print(f'Embedding dim  : {embed_dim}\n')

# Create an Embedding layer (randomly initialised)
torch.manual_seed(SEED)
embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)

# Look up vectors for a few words
demo_words = ['film', 'movi', 'terribl', 'brilliant', 'act', 'bore']
demo_words = [w for w in demo_words if w in word2idx]  # filter to vocab

idx_tensor = torch.tensor([word2idx[w] for w in demo_words])
vectors = embedding(idx_tensor)

print('Word vectors (first 6 dims):')
for w, vec in zip(demo_words, vectors):
    dims = ', '.join(f'{v:.3f}' for v in vec[:6].tolist())
    print(f'  {w:>12s} -> [{dims} ...]')

In [ ]:
# Cell 20: Cosine Similarity Between Word Pairs
from torch.nn.functional import cosine_similarity

def word_sim(w1: str, w2: str) -> float:
    '''Compute cosine similarity between two word embeddings.'''
    v1 = embedding(torch.tensor([word2idx[w1]]))
    v2 = embedding(torch.tensor([word2idx[w2]]))
    return cosine_similarity(v1, v2).item()

pairs_to_check = [
    ('film', 'movi'), ('film', 'bore'),
    ('brilliant', 'act'), ('terribl', 'bore'),
]
print('Cosine Similarity (random embeddings):\n')
for w1, w2 in pairs_to_check:
    if w1 in word2idx and w2 in word2idx:
        sim = word_sim(w1, w2)
        print(f'  sim({w1:>10s}, {w2:<10s}) = {sim:+.4f}')

print('\nNote: These are RANDOM embeddings, so similarities are arbitrary.')
print('After training (Word2Vec, GloVe), similar words cluster together.')

### 🗺️ PCA Projection of Word Vectors

Even with random embeddings, we can demonstrate the **workflow** for visualising word vectors in 2-D using PCA.

In [ ]:
# Cell 21: 2-D PCA of Word Vectors
from sklearn.decomposition import PCA

# Select a representative subset of words to plot
plot_words = [
    'film', 'movi', 'act', 'perform', 'direct',
    'brilliant', 'stun', 'beauti', 'except',
    'terribl', 'bore', 'aw', 'dull', 'disappoint',
    'love', 'great', 'best', 'plot', 'scene',
]
plot_words = [w for w in plot_words if w in word2idx]

idx_t = torch.tensor([word2idx[w] for w in plot_words])
vecs = embedding(idx_t).detach().numpy()

pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(vecs)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(coords[:, 0], coords[:, 1], s=100, c='steelblue', edgecolors='white', zorder=5)
for i, word in enumerate(plot_words):
    ax.annotate(word, (coords[i, 0] + 0.02, coords[i, 1] + 0.02),
               fontsize=11, fontweight='bold')

ax.set_title('2-D PCA Projection of Word Embeddings (Random Init)', fontsize=13)
ax.set_xlabel(f'PC 1 ({pca.explained_variance_ratio_[0]:.0%} variance)')
ax.set_ylabel(f'PC 2 ({pca.explained_variance_ratio_[1]:.0%} variance)')
ax.axhline(0, color='grey', lw=0.5, ls='--')
ax.axvline(0, color='grey', lw=0.5, ls='--')
plt.tight_layout()
plt.show()

print('After training (Word2Vec etc.), similar words would cluster together.')

---

# Part 5 — Introduction to Transformers (Conceptual)

---

### 🤖 What Are Transformers?

The **Transformer** architecture (Vaswani et al., 2017, *"Attention Is All You Need"*) revolutionised NLP by replacing recurrent layers with a **self-attention mechanism** that processes all tokens in parallel.

#### Self-Attention in a Nutshell

For each word in a sentence, self-attention computes a weighted combination of *all* other words, where the weights reflect relevance:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

This lets the model capture **long-range dependencies** far more efficiently than RNNs or LSTMs.

#### BERT vs. GPT

| | BERT | GPT |
|---|---|---|
| **Architecture** | Encoder only | Decoder only |
| **Pre-training** | Masked language modelling (fill in blanks) | Autoregressive (predict next word) |
| **Best for** | Classification, NER, Q&A | Text generation, chat, summarisation |
| **Directionality** | Bidirectional | Left-to-right |

#### The Hugging Face 🤗 Ecosystem

[Hugging Face](https://huggingface.co/) provides:
- **`transformers`** — thousands of pre-trained models (BERT, GPT-2, T5, LLaMA, …)
- **`datasets`** — curated NLP benchmarks
- **`tokenizers`** — fast, trainable tokenizers
- **`accelerate`** / **`peft`** — efficient fine-tuning on consumer hardware

```python
# Example (not run here — requires downloading a model)
from transformers import pipeline

classifier = pipeline('sentiment-analysis')
classifier('I absolutely loved this movie!')
# [{'label': 'POSITIVE', 'score': 0.9998}]
```

### 🚀 Next Steps

| Resource | Link |
|----------|------|
| **Hugging Face NLP Course** | [huggingface.co/learn/nlp-course](https://huggingface.co/learn/nlp-course) |
| **Fine-tuning BERT for Classification** | [Hugging Face fine-tuning tutorial](https://huggingface.co/docs/transformers/training) |
| **Stanford CS224N** | [web.stanford.edu/class/cs224n](https://web.stanford.edu/class/cs224n/) |
| **spaCy Advanced NLP** | [course.spacy.io](https://course.spacy.io/) |

---

# Part 6 — Wrap-up

---

### 📋 Summary — The NLP Pipeline

```
  Raw Text
     |
     v
  +----------------------+
  |  1. Preprocessing    |  Tokenize -> Remove stopwords -> Stem/Lemmatize
  +----------+-----------+
             v
  +----------------------+
  |  2. Representation   |  Bag-of-Words / TF-IDF / Word Embeddings
  +----------+-----------+
             v
  +----------------------+
  |  3. Modelling        |  Naive Bayes / Logistic Regression / Transformers
  +----------+-----------+
             v
  +----------------------+
  |  4. Evaluation       |  Accuracy, Precision, Recall, F1, Confusion Matrix
  +----------------------+
```

**Key Takeaways:**
1. **Text preprocessing** (tokenization, stopwords, stemming) is essential — garbage in, garbage out.
2. **TF-IDF** is a simple yet powerful text representation that often beats raw word counts.
3. **Logistic Regression + TF-IDF** is a surprisingly strong baseline for many NLP tasks.
4. **Word embeddings** capture semantic relationships that BoW cannot.
5. **Transformers** (BERT, GPT) are the state of the art — but classical methods are still valuable for small datasets and interpretability.

### ✏️ Exercise: Build Your Own Sentiment Classifier

Use the following **10 new reviews** and build a complete sentiment-analysis pipeline from scratch.

**Steps:**
1. Define the reviews and labels below.
2. Build a `Pipeline` with `TfidfVectorizer` and a classifier of your choice.
3. Use cross-validation to estimate performance.
4. Print a classification report.

In [ ]:
# Cell 22: Exercise — Build Your Own Sentiment Classifier
exercise_reviews = [
    "A riveting thriller that kept me guessing until the very last scene.",        # positive
    "Stale humor and recycled jokes. A comedy that forgot to be funny.",           # negative
    "Beautiful animation and a touching story. Perfect for all ages.",             # positive
    "Confusing timeline and poorly developed characters. Very frustrating.",       # negative
    "The lead actor delivered a career-best performance. Truly moving.",           # positive
    "Dragged on forever with no real climax. I nearly fell asleep.",               # negative
    "A bold, original take on a classic story. Refreshingly creative.",            # positive
    "Terrible CGI and a nonsensical plot. Felt like a cash grab.",                 # negative
    "Uplifting and full of heart. Left the theater feeling inspired.",             # positive
    "Flat dialogue and zero chemistry between the leads. A total misfire.",        # negative
]

exercise_labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]

# --- YOUR CODE BELOW ---
# Hint: reuse clean_text() and the Pipeline pattern from Part 3
#
# exercise_pipe = Pipeline([
#     ('tfidf', TfidfVectorizer()),
#     ('clf', LogisticRegression(max_iter=1000)),
# ])
# ...
print('Exercise ready! Write your solution above.')

### 📚 Resources

| Resource | Link |
|----------|------|
| **Hugging Face Transformers** | [huggingface.co/docs/transformers](https://huggingface.co/docs/transformers) |
| **spaCy** | [spacy.io](https://spacy.io/) |
| **NLTK** | [nltk.org](https://www.nltk.org/) |
| **scikit-learn Text Tutorial** | [scikit-learn.org/stable/tutorial/text_analytics](https://scikit-learn.org/stable/tutorial/text_analytics/working_with_text_data.html) |
| **Stanford CS224N (NLP with Deep Learning)** | [web.stanford.edu/class/cs224n](https://web.stanford.edu/class/cs224n/) |
| **Speech and Language Processing (Jurafsky & Martin)** | [web.stanford.edu/~jurafsky/slp3](https://web.stanford.edu/~jurafsky/slp3/) |

---

*Tutorial created with ❤️ — Happy NLP-ing!*